## structured output
structured output ensures that a Large Language Model (LLM) returns its response in a predictable, predefined format(well structered) (such as JSON) rather than unstructured text

### pydantic 
Pydantic model provide the richest feature set with field validation,descriptions and nested structures
- Pydantic as the primary method to enforce structure. It allows you to define a schema with specific fields (like name, email, or phone number) that the LLM must follow.


In [15]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")     
model



ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001F64E37CB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F64E37D590>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [16]:
from pydantic import BaseModel,Field

class movie(BaseModel):
    title:str = Field(decription = "the title of the movie")
    hero:str = Field(description="the hero of the movie")
    year:int = Field(description="this year the movie was released")
    directior:str = Field(description="the director of the movie")
    rating:float = Field(description="the movie rating out of 10")

C:\Users\preet\AppData\Local\Temp\ipykernel_2056\1021980438.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  title:str = Field(decription = "the title of the movie")


In [17]:
model_with_structure = model.with_structured_output(movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001F64E37CB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F64E37D590>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': {'decription': 'the title of the movie', 'type': 'string'}, 'hero': {'description': 'the hero of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'directior': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': 'the movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'hero', 'year', 'directior', 'rating'], 'type': 'object'}

In [18]:
response = model.invoke("provide details about the movie Ironman")
print(response.content)


<think>
Here's a thinking process:

1.  **Understand User Request:** The user asked for "details about the movie Ironman". I need to provide comprehensive, accurate information about the film "Iron Man" (2008).

2.  **Identify Key Aspects to Cover:**
   - Title & Release Year
   - Director & Writers
   - Cast
   - Plot Summary
   - Production Details
   - Box Office & Reception
   - Significance/Impact
   - Awards & Nominations
   - Sequels/Spin-offs (briefly, as context)

3.  **Gather Facts (Internal Knowledge):**
   - *Title:* Iron Man
   - *Release Year:* 2008
   - *Director:* Jon Favreau
   - *Writers:* Mark Fergus, Hawk Ostby, Art Marcum, Matt Holloway (based on Marvel Comics characters by Stan Lee, Larry Lieber, Don Heck, Jack Kirby)
   - *Cast:* Robert Downey Jr. (Tony Stark/Iron Man), Gwyneth Paltrow (Pepper Potts), Jeff Bridges (Obadiah Stane), Terrence Howard (James "Rhodey" Rhodes), Shaun Toub (Yinsen), Leslie Bibb (Christine Everhart), Sam Rockwell (Justin Hammer)
   - *Pl

In [19]:
response = model_with_structure.invoke("provide details about the movie Ironman")
response

movie(title='Iron Man', hero='Tony Stark', year=2008, directior='Jon Favreau', rating=7.9)

### Message output alongside parsed structure



In [21]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A Movie with detailes."""
    title:str = Field(...,description="the titla of the movie")
    year :int = Field(...,description="the year the movie was released")
    director:str = Field(...,description="the director of the movie")
    rating:float = Field(...,description="the movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("provide details about the movie Ironman")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "provide details about the movie Ironman"\n   - Key entity: "Ironman" (likely refers to "Iron Man", the 2008 Marvel movie)\n   - Required information: title, year, director, rating (based on the available function schema)\n\n2.  **Identify Available Tools:**\n   - I have a function called `Movie` that takes:\n     - `title` (string, required)\n     - `year` (integer, required)\n     - `director` (string, required)\n     - `rating` (number, required)\n   - This function seems to be for *providing* or *retrieving* movie details, but the description says "A Movie with detailes." It\'s likely meant to be called with the known details to return/format them, or it\'s a mock function that expects me to fill in the known facts.\n\n3.  **Gather Required Information (Internal Knowledge):**\n   - Title: Iron Man (or Ironman as user typed, but I\'ll use

### nested structer

In [23]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    Budget:float | None = Field(None,description="budget in million USD")

model_with_structure = model.with_structured_output(MovieDetail)

response = model_with_structure.invoke("provide the details about the movie Iron man")
response

MovieDetail(title='Iron Man', year=2008, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Gwyneth Paltrow', role='Pepper Potts'), Actor(name='Jeff Bridges', role='Obadiah Stane / Iron Monger'), Actor(name='Terrence Howard', role="James 'Rhodey' Rhodes"), Actor(name='Jon Favreau', role='Happy Hogan')], genres=['Action', 'Adventure', 'Sci-Fi'], Budget=140.0)

### TypedDict
it ptovides a simpler alternative using python's build-in typing,ideal when you don't need runtime validation

In [24]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """ a movie with dict"""
    title:Annotated[str,...,"the title of the movie"]
    year:Annotated[int,...,"the year the movie is released"]
    director:Annotated[str,...,"the director of the movie"]
    rating:Annotated[float,...,"the movies rating out of 10"]

movie_with_typedDict = model.with_structured_output(MovieDict)
response = movie_with_typedDict.invoke("provite the detailes of Iron Man")
response



{'director': 'Jon Favreau', 'rating': 7.9, 'title': 'Iron Man', 'year': 2008}

None


### Data class

In [ ]:
# useing pydanitc
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """content information for a person"""
    name:str = Field(description="the name of the person")
    email:str = Field(description="the email address of the person")
    phone:str = Field(description="the phone number of the preson")

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    response_format = ContactInfo # auto-selects Providerstrategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contect info from :jone doe,john@example.com,(555) 123-4567"}]
})

print(result["structured_response"])

name='jone doe' email='john@example.com' phone='(555) 123-4567'


In [37]:
#using dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """contect info"""
    name:str
    email:str
    phone:str

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contect info from :jone doe,john@example.com,(555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='jone doe', email='john@example.com', phone='(555) 123-4567')
